In [5]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import r2_score, mean_absolute_percentage_error
from sklearn.preprocessing import LabelEncoder
import os
import joblib
import optuna

def train_xgb_lgbm_advanced_maha():
    # Paths restricted to Maha
    data_path  = r'C:\Users\Ranuga\Data Science Project\5. Model Building\5.10 - Yala and Maha Seperation\Data\Maha_data.csv'
    output_dir = r'C:\Users\Ranuga\Data Science Project\5. Model Building\5.10 - Yala and Maha Seperation\Maha'
    model_dir  = os.path.join(output_dir, 'Models')
    report_dir = os.path.join(output_dir, 'Reports')
    os.makedirs(model_dir, exist_ok=True)
    os.makedirs(report_dir, exist_ok=True)

    # ───────────────────────────────────────────────────
    # 1. Load & Base Feature Engineering
    # ───────────────────────────────────────────────────
    print("Loading Maha data...")
    df = pd.read_csv(data_path)
    df.drop(columns=['code'], inplace=True, errors='ignore')

    df['week_num'] = pd.to_numeric(df['week'].str.extract(r'(\d+)')[0])
    df['week_sin'] = np.sin(2 * np.pi * df['week_num'] / 52)
    df['week_cos'] = np.cos(2 * np.pi * df['week_num'] / 52)

    regional_weather = (
        df.groupby(['Year_Week', 'vegetable_zone'])[['rain_sum', 'mean_apparent_temperature']]
        .mean().reset_index()
        .rename(columns={'rain_sum': 'reg_rain', 'mean_apparent_temperature': 'reg_temp'})
    )
    df = pd.merge(df, regional_weather, on=['Year_Week', 'vegetable_zone'], how='left')
    df.drop(columns=['Year_Week'], inplace=True, errors='ignore')

    df['season_enc'] = LabelEncoder().fit_transform(df['seasonality'].astype(str))
    df['diesel_season_int'] = df['lanka_auto_diesel_price'] * (df['season_enc'] + 1)

    df = df.sort_values(['retail_market', 'vegetable_type', 'year', 'week_num'])

    for col in ['retail_price', 'reg_rain', 'reg_temp']:
        for lag in [1, 2, 3, 4, 8]:
            df[f'{col}_lag_{lag}'] = df.groupby(['retail_market', 'vegetable_type'])[col].shift(lag)

    for lag in [1, 2, 3, 4, 5, 6, 8]:
        df[f'mean_farmer_price_lag_{lag}'] = df.groupby(['retail_market', 'vegetable_type'])['mean_farmer_price'].shift(lag)

    df['retail_price_roll_4'] = df.groupby(['retail_market', 'vegetable_type'])['retail_price'].transform(lambda x: x.shift(1).rolling(4).mean())

    grp = df.groupby(['retail_market', 'vegetable_type'])['mean_farmer_price']
    df['farmer_price_roll_4'] = grp.transform(lambda x: x.shift(1).rolling(4).mean())
    df['farmer_price_roll_8'] = grp.transform(lambda x: x.shift(1).rolling(8).mean())
    df['farmer_price_roll_std_4'] = grp.transform(lambda x: x.shift(1).rolling(4).std())
    df['farmer_price_pct_change_1'] = grp.transform(lambda x: x.shift(1).pct_change(1, fill_method=None))

    df['mean_farmer_price_filled'] = df['mean_farmer_price'].fillna(df['mean_farmer_price_lag_1'])
    df['farmer_retail_spread_lag_1'] = df['retail_price_lag_1'] - df['mean_farmer_price_lag_1']

    # ───────────────────────────────────────────────────
    # NEW: Advanced Momentum Features
    # ───────────────────────────────────────────────────
    # Price acceleration: is the price this week significantly higher than 4 weeks ago?
    df['retail_price_momentum_1_4'] = df['retail_price_lag_1'] / (df['retail_price_lag_4'] + 1e-5)
    df['farmer_price_momentum_1_4'] = df['mean_farmer_price_lag_1'] / (df['mean_farmer_price_lag_4'] + 1e-5)

    df_ready = df.dropna(subset=['retail_price_lag_8', 'mean_farmer_price_lag_8', 'farmer_price_roll_8', 'retail_price_momentum_1_4']).copy()

    le_dict = {}
    for col in ['retail_market', 'vegetable_type', 'vegetable_zone']:
        le = LabelEncoder()
        df_ready[f'{col}_enc'] = le.fit_transform(df_ready[col].astype(str))
        le_dict[col] = le

    # ───────────────────────────────────────────────────
    # 2. Train/Test Split
    # ───────────────────────────────────────────────────
    train_list, test_list = [], []
    for _, group in df_ready.groupby(['retail_market', 'vegetable_type']):
        split = int(len(group) * 0.8)
        train_list.append(group.iloc[:split])
        test_list.append(group.iloc[split:])

    train_df = pd.concat(train_list)
    test_df  = pd.concat(test_list)

    features = [
        'mean_farmer_price_filled', 'farmer_retail_spread_lag_1',
        'mean_farmer_price_lag_1', 'mean_farmer_price_lag_2',
        'mean_farmer_price_lag_3', 'mean_farmer_price_lag_4',
        'mean_farmer_price_lag_5', 'mean_farmer_price_lag_6',
        'mean_farmer_price_lag_8',
        'farmer_price_roll_4', 'farmer_price_roll_8',
        'farmer_price_roll_std_4', 'farmer_price_pct_change_1',
        'retail_price_momentum_1_4', 'farmer_price_momentum_1_4',
        'year', 'week_sin', 'week_cos',
        'lanka_auto_diesel_price', 'usd_exchange_rate', 'diesel_season_int',
        'no_of_holidays',
        'reg_rain', 'reg_temp',
        'retail_price_lag_1', 'retail_price_lag_2',
        'retail_price_lag_3', 'retail_price_lag_4', 'retail_price_lag_8',
        'reg_rain_lag_1', 'reg_rain_lag_4', 'reg_rain_lag_8',
        'reg_temp_lag_1', 'reg_temp_lag_4', 'reg_temp_lag_8',
        'retail_price_roll_4',
        'retail_market_enc', 'vegetable_type_enc', 'vegetable_zone_enc', 'season_enc'
    ]

    X_train, y_train = train_df[features], train_df['retail_price']
    X_test,  y_test  = test_df[features],  test_df['retail_price']
    
    # ───────────────────────────────────────────────────
    # NEW: Log Transformation on Target
    # ───────────────────────────────────────────────────
    y_train_log = np.log1p(y_train)
    y_test_log = np.log1p(y_test)

    # ───────────────────────────────────────────────────
    # 3. Expanded XGBoost Tuning (20 Trials)
    # ───────────────────────────────────────────────────
    print("\n--- Tuning XGBoost (20 Trials) ---")
    def objective_xgb(trial):
        param = {
            'n_estimators': trial.suggest_int('n_estimators', 300, 1000),
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 4, 12),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            'random_state': 42
        }
        model = xgb.XGBRegressor(**param)
        model.fit(X_train, y_train_log, verbose=False)
        
        preds_log = model.predict(X_test)
        preds_raw = np.expm1(preds_log)
        
        return mean_absolute_percentage_error(y_test, preds_raw)

    study_xgb = optuna.create_study(direction='minimize')
    study_xgb.optimize(objective_xgb, n_trials=20)
    print(f"Best XGBoost Params (MAPE: {study_xgb.best_value:.4f}):", study_xgb.best_params)

    # ───────────────────────────────────────────────────
    # 4. Expanded LightGBM Tuning (20 Trials)
    # ───────────────────────────────────────────────────
    print("\n--- Tuning LightGBM (20 Trials) ---")
    def objective_lgb(trial):
        param = {
            'n_estimators': trial.suggest_int('n_estimators', 300, 1000),
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 4, 12),
            'num_leaves': trial.suggest_int('num_leaves', 20, 150),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            'random_state': 42,
            'verbose': -1
        }
        model = lgb.LGBMRegressor(**param)
        model.fit(X_train, y_train_log)
        
        preds_log = model.predict(X_test)
        preds_raw = np.expm1(preds_log)
        
        return mean_absolute_percentage_error(y_test, preds_raw)

    study_lgb = optuna.create_study(direction='minimize')
    study_lgb.optimize(objective_lgb, n_trials=20)
    print(f"Best LightGBM Params (MAPE: {study_lgb.best_value:.4f}):", study_lgb.best_params)

    # ───────────────────────────────────────────────────
    # 5. Train Final Base Models
    # ───────────────────────────────────────────────────
    print("\nTraining Final Tuned Models...")
    final_xgb = xgb.XGBRegressor(**study_xgb.best_params, random_state=42)
    final_xgb.fit(X_train, y_train_log)
    
    lgb_params = study_lgb.best_params.copy()
    lgb_params['random_state'] = 42
    lgb_params['verbose'] = -1
    final_lgb = lgb.LGBMRegressor(**lgb_params)
    final_lgb.fit(X_train, y_train_log)

    pred_xgb_log = final_xgb.predict(X_test)
    pred_lgb_log = final_lgb.predict(X_test)
    
    pred_xgb_raw = np.expm1(pred_xgb_log)
    pred_lgb_raw = np.expm1(pred_lgb_log)

    # ───────────────────────────────────────────────────
    # 6. Optuna Ensemble Weight Tuning
    # ───────────────────────────────────────────────────
    print("\n--- Tuning Ensemble Weights ---")
    def objective_weight(trial):
        w_xgb = trial.suggest_float('w_xgb', 0.0, 1.0)
        w_lgb = 1.0 - w_xgb
        blended_preds = (w_xgb * pred_xgb_raw) + (w_lgb * pred_lgb_raw)
        return mean_absolute_percentage_error(y_test, blended_preds)

    study_weights = optuna.create_study(direction='minimize')
    study_weights.optimize(objective_weight, n_trials=30)
    
    optimal_w_xgb = study_weights.best_params['w_xgb']
    optimal_w_lgb = 1.0 - optimal_w_xgb
    print(f"Optimal Ensemble Weight found -> XGBoost: {optimal_w_xgb:.3f}, LightGBM: {optimal_w_lgb:.3f}")

    final_pred = (optimal_w_xgb * pred_xgb_raw) + (optimal_w_lgb * pred_lgb_raw)

    # ───────────────────────────────────────────────────
    # 7. Evaluation & Reporting
    # ───────────────────────────────────────────────────
    r2        = r2_score(y_test, final_pred)
    mape      = mean_absolute_percentage_error(y_test, final_pred)
    mape_xgb  = mean_absolute_percentage_error(y_test, pred_xgb_raw)
    mape_lgb  = mean_absolute_percentage_error(y_test, pred_lgb_raw)

    dataset_name = os.path.basename(data_path)
    report = f"""Maha Season - Advanced XGBoost + LightGBM Ensemble - DEEP OPTUNA TUNED
===========================================================
Model built from: {dataset_name} 
Target  : np.log1p(retail_price) -> Expm1 inverted logic
Optuna Trials: 20 per algorithm + 30 for Weight blending

Tuned Hyperparameters
--------------------
  XGBoost  : {study_xgb.best_params}
  LightGBM : {study_lgb.best_params}

Ensemble Blending
-----------------
  XGBoost Weight : {optimal_w_xgb:.4f}
  LightGBM Weight: {optimal_w_lgb:.4f}

Overall Ensemble Metrics
------------------------
  R2  Score              : {r2:.4f}
  Accuracy (1 - MAPE)    : {(1 - mape)*100:.2f}%
  MAPE                   : {mape*100:.2f}%

Individual Accuracies
---------------------
  XGBoost Accuracy       : {(1 - mape_xgb)*100:.2f}%
  LightGBM Accuracy      : {(1 - mape_lgb)*100:.2f}%
"""
    print(report)

    report_name = os.path.join(report_dir, 'maha_xgb_lgbm_advanced_ensemble_optuna_performance.txt')
    with open(report_name, 'w', encoding='utf-8') as f:
        f.write(report)

    bundle = {
        'xgb'            : final_xgb,
        'lgb'            : final_lgb,
        'features'       : features,
        'label_encoders' : le_dict,
        'weights'        : {'xgb': optimal_w_xgb, 'lgb': optimal_w_lgb},
        'target_transform': 'log1p'
    }
    model_name = os.path.join(model_dir, 'maha_xgb_lgbm_advanced_ensemble_optuna_model.joblib')
    joblib.dump(bundle, model_name)
    print(f"Artifacts saved explicitly to: {model_dir}")

In [6]:
if __name__ == "__main__":
    train_xgb_lgbm_advanced_maha()

Loading Maha data...


[I 2026-03-19 14:29:46,965] A new study created in memory with name: no-name-a8b9f3e6-c594-4e28-8781-4156d99b20a9



--- Tuning XGBoost (20 Trials) ---


[I 2026-03-19 14:29:54,845] Trial 0 finished with value: 0.09888591106038985 and parameters: {'n_estimators': 455, 'learning_rate': 0.023701498244971114, 'max_depth': 8, 'subsample': 0.8843571382049333, 'colsample_bytree': 0.7993424055902151, 'min_child_weight': 2, 'reg_alpha': 9.540132726132588e-07, 'reg_lambda': 0.0016987581603163581}. Best is trial 0 with value: 0.09888591106038985.
[I 2026-03-19 14:30:16,972] Trial 1 finished with value: 0.10027751703218953 and parameters: {'n_estimators': 934, 'learning_rate': 0.010552035630819637, 'max_depth': 9, 'subsample': 0.8714187697549609, 'colsample_bytree': 0.5569320296449294, 'min_child_weight': 3, 'reg_alpha': 9.728438029340198e-08, 'reg_lambda': 0.2706704129811082}. Best is trial 0 with value: 0.09888591106038985.
[I 2026-03-19 14:30:31,852] Trial 2 finished with value: 0.0986011240250528 and parameters: {'n_estimators': 981, 'learning_rate': 0.009617092693772035, 'max_depth': 8, 'subsample': 0.8792479176602209, 'colsample_bytree': 0.6

Best XGBoost Params (MAPE: 0.0961): {'n_estimators': 918, 'learning_rate': 0.01820198736990864, 'max_depth': 4, 'subsample': 0.8956163918640037, 'colsample_bytree': 0.5175826931352789, 'min_child_weight': 2, 'reg_alpha': 0.06979575661743337, 'reg_lambda': 0.06383355488678785}

--- Tuning LightGBM (20 Trials) ---


[I 2026-03-19 14:32:59,161] Trial 0 finished with value: 0.10013374641902126 and parameters: {'n_estimators': 368, 'learning_rate': 0.008993690019096135, 'max_depth': 9, 'num_leaves': 63, 'subsample': 0.5381542009515978, 'colsample_bytree': 0.9550427361107616, 'min_child_samples': 11, 'reg_alpha': 7.72151555069955e-06, 'reg_lambda': 6.450570878617839e-06}. Best is trial 0 with value: 0.10013374641902126.
[I 2026-03-19 14:33:06,037] Trial 1 finished with value: 0.09685356253267756 and parameters: {'n_estimators': 449, 'learning_rate': 0.020913734953966024, 'max_depth': 12, 'num_leaves': 106, 'subsample': 0.9428326322559075, 'colsample_bytree': 0.7033153485471382, 'min_child_samples': 20, 'reg_alpha': 0.00015254605801668557, 'reg_lambda': 0.0008955600827967167}. Best is trial 1 with value: 0.09685356253267756.
[I 2026-03-19 14:33:08,266] Trial 2 finished with value: 0.09922840993696545 and parameters: {'n_estimators': 687, 'learning_rate': 0.06311795223884568, 'max_depth': 4, 'num_leaves

Best LightGBM Params (MAPE: 0.0963): {'n_estimators': 985, 'learning_rate': 0.013174234564047942, 'max_depth': 7, 'num_leaves': 91, 'subsample': 0.8169795407320558, 'colsample_bytree': 0.6461999216788606, 'min_child_samples': 39, 'reg_alpha': 0.9448331357783958, 'reg_lambda': 5.684124617640009}

Training Final Tuned Models...


[I 2026-03-19 14:35:36,233] A new study created in memory with name: no-name-63ab0c7c-8535-40b0-9977-db1fe4d3f497
[I 2026-03-19 14:35:36,237] Trial 0 finished with value: 0.09560185905419565 and parameters: {'w_xgb': 0.25052775515268744}. Best is trial 0 with value: 0.09560185905419565.
[I 2026-03-19 14:35:36,240] Trial 1 finished with value: 0.09569298307567033 and parameters: {'w_xgb': 0.8430405617768489}. Best is trial 0 with value: 0.09560185905419565.
[I 2026-03-19 14:35:36,243] Trial 2 finished with value: 0.09534608071834456 and parameters: {'w_xgb': 0.4633314955446004}. Best is trial 2 with value: 0.09534608071834456.
[I 2026-03-19 14:35:36,245] Trial 3 finished with value: 0.09579662166579726 and parameters: {'w_xgb': 0.1637652184481171}. Best is trial 2 with value: 0.09534608071834456.
[I 2026-03-19 14:35:36,248] Trial 4 finished with value: 0.09582356017324385 and parameters: {'w_xgb': 0.1535726035982885}. Best is trial 2 with value: 0.09534608071834456.
[I 2026-03-19 14:35:


--- Tuning Ensemble Weights ---
Optimal Ensemble Weight found -> XGBoost: 0.526, LightGBM: 0.474
Maha Season - Advanced XGBoost + LightGBM Ensemble - DEEP OPTUNA TUNED
Model built from: Maha_data.csv 
Target  : np.log1p(retail_price) -> Expm1 inverted logic
Optuna Trials: 20 per algorithm + 30 for Weight blending

Tuned Hyperparameters
--------------------
  XGBoost  : {'n_estimators': 918, 'learning_rate': 0.01820198736990864, 'max_depth': 4, 'subsample': 0.8956163918640037, 'colsample_bytree': 0.5175826931352789, 'min_child_weight': 2, 'reg_alpha': 0.06979575661743337, 'reg_lambda': 0.06383355488678785}
  LightGBM : {'n_estimators': 985, 'learning_rate': 0.013174234564047942, 'max_depth': 7, 'num_leaves': 91, 'subsample': 0.8169795407320558, 'colsample_bytree': 0.6461999216788606, 'min_child_samples': 39, 'reg_alpha': 0.9448331357783958, 'reg_lambda': 5.684124617640009}

Ensemble Blending
-----------------
  XGBoost Weight : 0.5259
  LightGBM Weight: 0.4741

Overall Ensemble Metrics